# Common Libraries

In [1]:
import os, sys, shutil, mediapy, copy
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
from ipywidgets import interact, IntSlider, FloatSlider, Layout, Button, Output, VBox, HBox

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from video_tool import save_video

# Params

In [32]:
mp_model_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41/temp_cvt3.xml"
nmp_model_path = "/mnt/ext1/seojin/temp/MOBL_ARMS_41_mp/temp_cvt3.xml"

simulation_interval = 0.005
is_video = True
fps = 1
n_step = 300

# Initialize Model

In [33]:
mp_model = mujoco.MjModel.from_xml_path(mp_model_path)
mp_model.opt.timestep = simulation_interval
mp_data = mujoco.MjData(mp_model)
mujoco.mj_forward(mp_model, mp_data)

nmp_model = mujoco.MjModel.from_xml_path(nmp_model_path)
nmp_model.opt.timestep = simulation_interval
nmp_data = mujoco.MjData(nmp_model)
mujoco.mj_forward(nmp_model, nmp_data)

# Render

In [34]:
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_WORLD

# Renderer
mp_renderer = mujoco.Renderer(mp_model, height=400, width=600)
nmp_renderer = mujoco.Renderer(nmp_model, height=400, width=600)

# Camera
mp_camera = mujoco.MjvCamera()
mp_camera.azimuth = 180
mp_camera.elevation = 0
mp_camera.distance = 1.8

nmp_camera = mujoco.MjvCamera()
nmp_camera.azimuth = 180
nmp_camera.elevation = 0
nmp_camera.distance = 1.8

In [35]:
# Initialization
mp_data.qpos[:] = np.zeros_like(mp_data.qpos)
nmp_data.qpos[:] = np.zeros_like(nmp_data.qpos)
mujoco.mj_forward(mp_model, mp_data)
mujoco.mj_forward(nmp_model, nmp_data)

# Dummy
frames = []

# UI
output_area = Output()
apply_btn = Button(description = "Apply", button_style = "success")
def make_frame():
    mp_renderer.update_scene(
        mp_data,
        camera=mp_camera,
        scene_option=scene_option
    )

    nmp_renderer.update_scene(
        nmp_data,
        camera=nmp_camera,
        scene_option=scene_option
    )
    img1 = mp_renderer.render().copy()
    img2 = nmp_renderer.render().copy()
    if img1.shape[0] != img2.shape[0]:
        h = min(img1.shape[0], img2.shape[0])
        img1 = img1[:h]
        img2 = img2[:h]
    frame = np.concatenate([img1, img2], axis=1)
    return frame
    
def show_frame():
    frame = copy.deepcopy(make_frame())
    if is_video:
        frames.append(frame)
    else:
        with output_area:
            output_area.clear_output(wait=True)
            plt.figure(figsize=(8, 6))
            plt.imshow(frame)
            plt.axis("off")
            plt.tight_layout()
            plt.show()

for _ in range(n_step):
    mujoco.mj_step(mp_model, mp_data)
    mujoco.mj_step(nmp_model, nmp_data)
    show_frame()

In [36]:
mediapy.show_video(frames)

This browser does not support the video tag.


In [37]:
save_video(np.array(frames), fps = 30, output_path = "/mnt/ext1/seojin/temp/mp_nmp_sim.mp4", is_progress_bar = True)

MoviePy - Building video /mnt/ext1/seojin/temp/mp_nmp_sim.mp4.
MoviePy - Writing video /mnt/ext1/seojin/temp/mp_nmp_sim.mp4



MoviePy - Done !
MoviePy - video ready /mnt/ext1/seojin/temp/mp_nmp_sim.mp4
save: /mnt/ext1/seojin/temp/mp_nmp_sim.mp4
